# ARGweaver Pipeline: 1000 Genomes → Newick Trees

This notebook documents the full pipeline from a raw 1000 Genomes Phase 3 VCF to per-sample ARGweaver Newick tree files ready for phylodyn analysis in R.

## Assumed inputs
- `ALL.chr2.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz` — full chr2 VCF
- `ALL.chr2.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz.tbi` — tabix index
- `integrated_call_samples_v3.20130502.ALL.panel` — sample population panel

## Pipeline steps
1. Download panel file and select samples
1.5. Download genetic map
2. Subset VCF to region and population
2.5. Estimate population parameters
3. Convert phased VCF → ARGweaver `.sites` format
4. Run `arg-sample` MCMC inference
5. Convert `.smc.gz` posterior samples → Newick `.tree` files + coordinate `.csv` files

## Requirements
```
conda install -c bioconda bcftools tabix
conda install -c bioconda argweaver
pip install tskit
```

## 0. Configuration — edit these values before running

In [48]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR = Path(".")  # directory containing the VCF files; change if needed

VCF = BASE_DIR / "ALL.chr2.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"

# Organized storage layout
DATA_DIR = BASE_DIR / "data"
INPUTS_DIR = DATA_DIR / "inputs"
METADATA_DIR = INPUTS_DIR / "metadata"
SAMPLE_LIST_DIR = INPUTS_DIR / "sample_lists"
RECOMB_MAP_DIR = INPUTS_DIR / "recomb_maps"
RUNS_DIR = DATA_DIR / "runs"
RESULTS_DIR = DATA_DIR / "results"

# ── Region ─────────────────────────────────────────────────────────────────────
CHROM      = "2"           # chromosome name as it appears in the VCF (no 'chr' prefix for b37)
REGION_START = 130_000_000  # bp, inclusive
REGION_END   = 140_000_000  # bp, inclusive
REGION = f"{CHROM}:{REGION_START}-{REGION_END}"

# ── Population & sample size ───────────────────────────────────────────────────
POPULATION   = "CEU"   # 1000G population code
N_SAMPLES    = 20      # number of individuals (→ 2*N_SAMPLES haplotypes)

# ── ARGweaver model parameters ─────────────────────────────────────────────────
POP_SIZE     = 10_000   # diploid effective population size
MUT_RATE     = 1.25e-8  # mutations per site per generation
RECOMB_RATE  = 1.0e-8   # recombinations per site per generation
NTIMES       = 30       # number of discrete time steps
MAXTIME      = 200_000  # maximum time in generations
COMPRESS     = 20       # sequence compression factor (bp per block)
MCMC_ITERS   = 300      # total MCMC iterations
SAMPLE_STEP  = 10       # write ARG every N iterations → MCMC_ITERS/SAMPLE_STEP output files

# ── Run/output directories ─────────────────────────────────────────────────────
RUN_TAG = f"chr{CHROM}_{REGION_START//1_000_000}_{REGION_END//1_000_000}Mb_{POPULATION}{N_SAMPLES}"
OUTDIR = RUNS_DIR / f"argweaver_{RUN_TAG}"
INTERMEDIATE_DIR = OUTDIR / "intermediate"
ARG_SAMPLES_DIR = OUTDIR / "smc"
TREES_DIR = OUTDIR / "trees"
CSV_DIR = RESULTS_DIR / RUN_TAG / "breakpoints"

for d in [
    METADATA_DIR,
    SAMPLE_LIST_DIR,
    RECOMB_MAP_DIR,
    INTERMEDIATE_DIR,
    ARG_SAMPLES_DIR,
    TREES_DIR,
    CSV_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Derived filenames
PANEL_FILE = METADATA_DIR / "integrated_call_samples_v3.20130502.ALL.panel"
SAMPLE_LIST = SAMPLE_LIST_DIR / f"{POPULATION}_{N_SAMPLES}.txt"
SUBSET_VCF = INTERMEDIATE_DIR / f"chr{CHROM}_{REGION_START//1_000_000}_{REGION_END//1_000_000}Mb_{POPULATION}{N_SAMPLES}.vcf.gz"
SITES_FILE = INTERMEDIATE_DIR / f"chr{CHROM}_{POPULATION}{N_SAMPLES}.sites"
ARG_PREFIX = str(ARG_SAMPLES_DIR / POPULATION.lower())

print("Configuration:")
print(f"  VCF:              {VCF}")
print(f"  Region:           {REGION}")
print(f"  Population:       {POPULATION} ({N_SAMPLES} individuals = {2*N_SAMPLES} haplotypes)")
print(f"  Run dir:          {OUTDIR}")
print(f"  Intermediate dir: {INTERMEDIATE_DIR}")
print(f"  SMC dir:          {ARG_SAMPLES_DIR}")
print(f"  Tree dir:         {TREES_DIR}")
print(f"  CSV dir:          {CSV_DIR}")


Configuration:
  VCF:              ALL.chr2.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
  Region:           2:130000000-140000000
  Population:       CEU (20 individuals = 40 haplotypes)
  Run dir:          data/runs/argweaver_chr2_130_140Mb_CEU20
  Intermediate dir: data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate
  SMC dir:          data/runs/argweaver_chr2_130_140Mb_CEU20/smc
  Tree dir:         data/runs/argweaver_chr2_130_140Mb_CEU20/trees
  CSV dir:          data/results/chr2_130_140Mb_CEU20/breakpoints


## 1. Download panel file and select samples

In [49]:
import subprocess

# Download panel file if not present
if not PANEL_FILE.exists():
    print("Downloading panel file...")
    panel_url = (
        "https://bochet.gcc.biostat.washington.edu/beagle/1000_Genomes_phase3_v5a/"
        "sample_info/integrated_call_samples_v3.20130502.ALL.panel"
    )
    subprocess.run(["curl", "-L", "-o", str(PANEL_FILE), panel_url], check=True)
    print(f"Saved to {PANEL_FILE}")
else:
    print(f"Panel file already exists: {PANEL_FILE}")

# Extract sample IDs for the chosen population
samples = []
with open(PANEL_FILE) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2 and parts[1] == POPULATION:
            samples.append(parts[0])

print(f"Found {len(samples)} {POPULATION} individuals in panel")

# Take the first N_SAMPLES
selected = samples[:N_SAMPLES]
with open(SAMPLE_LIST, "w") as f:
    f.write("\n".join(selected) + "\n")

print(f"Selected {len(selected)} individuals → {SAMPLE_LIST}")
print("First few:", selected[:5])

Panel file already exists: data/inputs/metadata/integrated_call_samples_v3.20130502.ALL.panel
Found 99 CEU individuals in panel
Selected 20 individuals → data/inputs/sample_lists/CEU_20.txt
First few: ['NA06984', 'NA06985', 'NA06986', 'NA06989', 'NA06994']


## 1.5 Download the 1000 Genomes Phase 3 genetic map

In [50]:
import subprocess
import pandas as pd
from pathlib import Path

# Download the 1000 Genomes Phase 3 genetic map (most recent)
map_url = f"http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/working/20130507_omni_recombination_rates/{POPULATION}_omni_recombination_20130507.tar"
map_tar = RECOMB_MAP_DIR / f"{POPULATION}_omni_recombination_20130507.tar"

if not map_tar.exists():
    print("Downloading recombination map...")
    subprocess.run(["curl", "-o", str(map_tar), map_url], check=True)
    print("Extracting...")
    subprocess.run(["tar", "-xf", str(map_tar), "-C", str(RECOMB_MAP_DIR)], check=True)
else:
    print("Recombination map already downloaded")


Recombination map already downloaded


In [51]:
import gzip

# Read the genetic map
# Format: Position(bp)  Rate(cM/Mb)  Map(cM)
map_file = RECOMB_MAP_DIR / f"{POPULATION}" / f"{POPULATION}-2-final.txt.gz"
print(f"Map file: {map_file}")

data = []
with gzip.open(map_file, 'rt') as f:
    next(f)  # skip header
    for line in f:
        parts = line.strip().split()
        pos = int(parts[0])
        rate_cM_per_Mb = float(parts[1])
        # Convert cM/Mb to per-bp-per-gen recombination rate
        # 1 cM/Mb = 1e-8 per bp per gen (approximately, assuming 1 cM = 1% recombination)
        rate_per_bp = rate_cM_per_Mb * 1e-8
        data.append((pos, rate_per_bp))

df = pd.DataFrame(data, columns=['pos', 'rate'])

# Filter to your region
region_map = df[(df['pos'] >= REGION_START) & (df['pos'] <= REGION_END)]

print(f"Map points in region: {len(region_map)}")
print(f"Rate range: {region_map['rate'].min():.2e} – {region_map['rate'].max():.2e}")
print(f"Mean rate:  {region_map['rate'].mean():.2e} per bp per gen")

# Show first few entries
print("First few map entries:")
print(region_map.head(10))


Map file: data/inputs/recomb_maps/CEU/CEU-2-final.txt.gz
Map points in region: 4438
Rate range: 0.00e+00 – 1.31e-06
Mean rate:  1.77e-08 per bp per gen
First few map entries:
             pos          rate
69671  130002588  1.047371e-09
69672  130003278  1.098717e-09
69673  130003549  1.177042e-09
69674  130003580  1.223166e-09
69675  130004192  1.328686e-09
69676  130005694  1.598253e-09
69677  130007148  2.391897e-08
69678  130007750  2.784019e-08
69679  130007996  2.682806e-08
69680  130008425  2.195259e-09


In [52]:
# Create bedGraph format
# Each line spans from one map point to the next
bedgraph_file = INTERMEDIATE_DIR / f"recomb_map_chr{CHROM}_{REGION_START}_{REGION_END}.bedgraph"

with open(bedgraph_file, 'w') as f:
    positions = region_map['pos'].values
    rates = region_map['rate'].values
    
    # First interval: region start to first map point
    if positions[0] > REGION_START:
        f.write(f"{CHROM}\t{REGION_START}\t{positions[0]}\t{rates[0]:.10e}\n")
    
    # Intervals between map points
    for i in range(len(positions) - 1):
        f.write(f"{CHROM}\t{positions[i]}\t{positions[i+1]}\t{rates[i]:.10e}\n")
    
    # Last interval: last map point to region end
    if positions[-1] < REGION_END:
        f.write(f"{CHROM}\t{positions[-1]}\t{REGION_END}\t{rates[-1]:.10e}\n")

print(f"\nWrote recombination map to: {bedgraph_file}")
print(f"Use in ARGweaver with: --recombmap {bedgraph_file}")


Wrote recombination map to: data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/recomb_map_chr2_130000000_140000000.bedgraph
Use in ARGweaver with: --recombmap data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/recomb_map_chr2_130000000_140000000.bedgraph


## 2. Subset VCF to region and population (bcftools)

In [53]:
import subprocess

if SUBSET_VCF.exists():
    print(f"Subset VCF already exists: {SUBSET_VCF}")
else:
    print(f"Subsetting VCF to {REGION}, population {POPULATION}...")
    cmd = [
        "bcftools", "view",
        "-r", REGION,
        "-S", str(SAMPLE_LIST),
        "-m2", "-M2",       # biallelic only
        "-v", "snps",        # SNPs only
        "-Oz",               # output bgzipped VCF
        "-o", str(SUBSET_VCF),
        str(VCF)
    ]
    subprocess.run(cmd, check=True)
    print("Indexing subset VCF...")
    subprocess.run(["bcftools", "index", "-t", str(SUBSET_VCF)], check=True)
    print("Done.")

# Sanity check: count variants
result = subprocess.run(
    ["bcftools", "view", str(SUBSET_VCF), "--no-header", "-H"],
    capture_output=True, text=True
)
n_variants = result.stdout.count("\n")
print(f"Variants in subset VCF: {n_variants:,}")
if n_variants == 0:
    raise RuntimeError(
        "Subset VCF is empty! Check that sample IDs in SAMPLE_LIST "
        "match the VCF column names exactly (run: bcftools query -l <VCF> | head)"
    )

Subsetting VCF to 2:130000000-140000000, population CEU...
Indexing subset VCF...
Done.
Variants in subset VCF: 279,824


## 2.5 Estimate parameters

In [54]:
# ─── Estimate parameters from data (CORRECTED) ────────────────────────────────
import allel
import numpy as np

print("Estimating parameters from data...")

# Load genotypes
callset = allel.read_vcf(str(SUBSET_VCF))
gt = allel.GenotypeArray(callset['calldata/GT'])
pos = callset['variants/POS']

n_haps = gt.n_samples * 2
ac = gt.count_alleles()

# Count segregating sites
is_seg = ac.is_segregating()
S = np.sum(is_seg)
L = REGION_END - REGION_START

# ─── Method 1: Watterson's estimator ──────────────────────────────────────────
a_n = sum(1/i for i in range(1, n_haps))
theta_W = S / a_n

mu = 1.25e-8  # per site per generation
Ne_watterson = theta_W / (4 * mu * L)

# ─── Method 2: Nucleotide diversity (π) — CORRECTED ───────────────────────────
# Compute pairwise differences correctly
# π = (sum over all sites of heterozygosity) / L
# heterozygosity at a site = 2 * p * (1-p) * n/(n-1) where p = allele freq

def nucleotide_diversity(ac):
    """
    Compute nucleotide diversity (π) from allele counts.
    π = average number of pairwise differences per site.
    """
    n = ac.sum(axis=1)  # total alleles per site (should be constant)
    # For biallelic sites: heterozygosity = 2*p*q * n/(n-1)
    # where p = freq of allele 0, q = freq of allele 1
    p = ac[:, 0] / n
    q = 1 - p
    # Expected pairwise differences per site
    het = 2 * p * q * n / (n - 1)
    return het

pi_per_site = nucleotide_diversity(ac)
pi_total = np.sum(pi_per_site)  # sum over all variant sites
pi_mean = pi_total / L           # average over entire sequence length

theta_pi = pi_mean * L
Ne_pi = theta_pi / (4 * mu * L)

print(f"\nSegregating sites:       {S:,}")
print(f"Sequence length:         {L:,} bp")
print(f"Sample size:             {n_haps} haplotypes")
print(f"\nWatterson's θ:           {theta_W:.2f}")
print(f"  → Estimated Ne:        {Ne_watterson:,.0f}")
print(f"\nNucleotide diversity π:  {pi_mean:.6f} per site")
print(f"  → Estimated Ne:        {Ne_pi:,.0f}")

# Sanity check: θ_W and θ_π should be similar (within 2x)
ratio = theta_pi / theta_W if theta_W > 0 else 0
print(f"\nθ_π / θ_W ratio:         {ratio:.2f}")
if ratio < 0.5 or ratio > 2.0:
    print("  ⚠️  WARNING: Large discrepancy suggests demographic history")
    print("     (e.g., recent bottleneck or expansion)")
    print(f"     Recommend using Ne ≈ {int(round(Ne_watterson, -3)):,}")
else:
    print(f"  ✓ Estimates agree. Recommend Ne ≈ {int(round(Ne_pi, -3)):,}")

# Optionally auto-update POP_SIZE
POP_SIZE = int(round(Ne_pi, -3))

Estimating parameters from data...

Segregating sites:       27,721
Sequence length:         10,000,000 bp
Sample size:             40 haplotypes

Watterson's θ:           6517.16
  → Estimated Ne:        13,034

Nucleotide diversity π:  0.000687 per site
  → Estimated Ne:        13,744

θ_π / θ_W ratio:         1.05
  ✓ Estimates agree. Recommend Ne ≈ 14,000


## 3. Convert phased VCF → ARGweaver `.sites` format

ARGweaver does not accept VCF directly (and ignores phasing if it did). 
The `.sites` format encodes only variant positions with alleles for each haplotype.

In [55]:
import gzip

def vcf_to_sites(vcf_gz_path, chrom, start, end, out_path):
    """
    Convert a phased, bgzipped VCF to ARGweaver .sites format.
    Only biallelic SNPs with full phasing (| separator) are included.
    Sites that are monomorphic after subsetting are dropped.
    """
    opener = gzip.open if str(vcf_gz_path).endswith(".gz") else open

    with opener(vcf_gz_path, "rt") as fh:
        samples = []
        hap_names = []
        rows = []

        for line in fh:
            if line.startswith("##"):
                continue

            if line.startswith("#CHROM"):
                samples = line.strip().split("\t")[9:]
                hap_names = []
                for s in samples:
                    hap_names += [f"{s}_1", f"{s}_2"]
                continue

            fields = line.strip().split("\t")
            pos = int(fields[1])
            ref, alt = fields[3], fields[4]

            # Skip non-biallelic, indels, or out-of-region sites
            if len(ref) != 1 or len(alt) != 1 or "," in alt:
                continue
            if pos < start or pos > end:
                continue

            gts = []
            skip = False
            for g in fields[9:]:
                gt = g.split(":")[0]
                if "|" not in gt:
                    skip = True  # unphased genotype — skip this site
                    break
                a1, a2 = gt.split("|")
                gts.append(ref if a1 == "0" else alt)
                gts.append(ref if a2 == "0" else alt)

            if skip:
                continue
            if len(gts) != len(hap_names):
                continue
            if len(set(gts)) == 1:
                continue  # invariant after subsetting

            rows.append((pos, "".join(gts)))

    with open(out_path, "w") as out:
        out.write("NAMES\t" + "\t".join(hap_names) + "\n")
        out.write(f"REGION\t{chrom}\t{start}\t{end}\n")
        for pos, alleles in rows:
            out.write(f"{pos}\t{alleles}\n")

    return len(rows), hap_names


if SITES_FILE.exists():
    print(f".sites file already exists: {SITES_FILE}")
    # Count SNPs
    with open(SITES_FILE) as f:
        n_sites = sum(1 for line in f if not line.startswith(("NAMES", "REGION")))
    print(f"Contains {n_sites:,} variant sites")
else:
    print("Converting VCF to .sites format...")
    n_sites, hap_names = vcf_to_sites(
        SUBSET_VCF, CHROM, REGION_START, REGION_END, SITES_FILE
    )
    print(f"Written {n_sites:,} variant sites to {SITES_FILE}")
    print(f"Haplotypes: {len(hap_names)} ({len(hap_names)//2} individuals)")
    print("First few haplotype names:", hap_names[:6])

# Final sanity check
with open(SITES_FILE) as f:
    header = [next(f) for _ in range(3)]
print("\nSites file header:")
for h in header:
    print(" ", h[:120].strip())

Converting VCF to .sites format...
Written 27,721 variant sites to data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/chr2_CEU20.sites
Haplotypes: 40 (20 individuals)
First few haplotype names: ['NA06984_1', 'NA06984_2', 'NA06985_1', 'NA06985_2', 'NA06986_1', 'NA06986_2']

Sites file header:
  NAMES	NA06984_1	NA06984_2	NA06985_1	NA06985_2	NA06986_1	NA06986_2	NA06989_1	NA06989_2	NA06994_1	NA06994_2	NA07000_1	NA07
  REGION	2	130000000	140000000
  130000040	CCCCCCCCGCCCCCCCCCCCCCCCCCCCGCCCCCCCCCCC


## 4. Run `arg-sample` (ARGweaver MCMC)

This is the main inference step. For a 10 Mb region with 40 haplotypes and 200 iterations, expect **~1 hour** runtime.

Output files: `{ARG_PREFIX}.{0,10,20,...,200}.smc.gz` — one ARG posterior sample per file.

In [56]:
import subprocess
import os

# Check if already run (look for at least the final sample file)
final_smc = Path(f"{ARG_PREFIX}.{MCMC_ITERS}.smc.gz")

if final_smc.exists():
    print(f"ARGweaver output already exists: {final_smc}")
    print("Delete output files and rerun this cell to redo inference.")
else:
    print("Running arg-sample...")
    print(f"  Sites file:  {SITES_FILE}")
    print(f"  Output:      {ARG_PREFIX}.*.smc.gz")
    print(f"  Iterations:  {MCMC_ITERS}  (writing every {SAMPLE_STEP})")
    print(f"  Expected output files: {MCMC_ITERS // SAMPLE_STEP + 1}")
    print()

    cmd = [
        "arg-sample",
        "--sites",    str(SITES_FILE),
        "--region",   f"{REGION_START}-{REGION_END}",
        "-N",         str(POP_SIZE),
        "-m",         str(MUT_RATE),
        # "-r",         str(RECOMB_RATE),
        "--recombmap", str(bedgraph_file),  # ← ADD THIS LINE
        "--ntimes",   str(NTIMES),
        "--maxtime",  str(MAXTIME),
        "-c",         str(COMPRESS),
        "-n",         str(MCMC_ITERS),
        "--sample-step", str(SAMPLE_STEP),
        "-o",         ARG_PREFIX,
    ]

    print("Command:")
    print(" ".join(cmd))
    print()

    # Stream output live so you can watch progress
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()

    if process.returncode != 0:
        raise RuntimeError(f"arg-sample failed with return code {process.returncode}")
    print("\narg-sample completed successfully.")

# List output files
smc_files = sorted(ARG_SAMPLES_DIR.glob(f"{POPULATION.lower()}.*.smc.gz"))
print(f"\nFound {len(smc_files)} .smc.gz files:")
for f in smc_files[:5]:
    print(f"  {f.name}")
if len(smc_files) > 5:
    print(f"  ... and {len(smc_files)-5} more")

Running arg-sample...
  Sites file:  data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/chr2_CEU20.sites
  Output:      data/runs/argweaver_chr2_130_140Mb_CEU20/smc/ceu.*.smc.gz
  Iterations:  300  (writing every 10)
  Expected output files: 31

Command:
arg-sample --sites data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/chr2_CEU20.sites --region 130000000-140000000 -N 14000 -m 1.25e-08 --recombmap data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/recomb_map_chr2_130000000_140000000.bedgraph --ntimes 30 --maxtime 200000 -c 20 -n 300 --sample-step 10 -o data/runs/argweaver_chr2_130_140Mb_CEU20/smc/ceu

arg-sample 0.8.1
start time: Mon Feb 16 19:08:49 2026
command: arg-sample --sites data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/chr2_CEU20.sites --region 130000000-140000000 -N 14000 -m 1.25e-08 --recombmap data/runs/argweaver_chr2_130_140Mb_CEU20/intermediate/recomb_map_chr2_130000000_140000000.bedgraph --ntimes 30 --maxtime 200000 -c 20 -n 300 --sample-step 10 -o dat

## 5. Convert `.smc.gz` → Newick `.tree` files + coordinate `.csv` files

Each `.smc.gz` file is one posterior ARG sample, represented as a series of local trees in NHX-annotated Newick format. We strip the NHX annotations, replace integer leaf indices with sample names, and write:
- `.tree` — one Newick string per line (one per non-recombining block)
- `_breaks.csv` — genomic coordinates of each tree interval (absolute positions)

These files are ready for phylodyn in R.

In [57]:
import gzip
import re
import csv
from pathlib import Path


def clean_newick(nhx, sample_names):
    """
    Convert ARGweaver NHX Newick to plain Newick:
      - Replace integer leaf indices with sample names
      - Remove internal node integer labels
      - Strip [&&NHX:...] annotations
    """
    n = len(sample_names)

    def replace_leaf(m):
        idx = int(m.group(1))
        return m.group(0).replace(m.group(1), sample_names[idx], 1) if idx < n else m.group(0)

    # Leaves are digit sequences preceded by ( or ,
    s = re.sub(r'(?<=[(,])(\d+)(?=:)', replace_leaf, nhx)
    # Remove internal node labels (digits immediately after closing paren)
    s = re.sub(r'\)(\d+)(?=:|\[|,|\)|;)', ')', s)
    # Strip NHX annotations
    s = re.sub(r'\[&&NHX:[^\]]*\]', '', s)
    return s.strip()


def smc_to_newick_records(smc_gz_path, sample_names):
    """
    Parse one .smc.gz file and return a list of tree records.
    Positions are absolute genomic coordinates (not relative to region start).
    """
    opener = gzip.open if str(smc_gz_path).endswith(".gz") else open

    with opener(smc_gz_path, "rt") as f:
        lines = f.readlines()

    region_parts = lines[1].strip().split("\t")
    seq_start = int(region_parts[2])
    seq_end   = int(region_parts[3])

    records = []
    for tree_index, line in enumerate(lines[2:]):
        if not line.startswith("TREE"):
            continue
        parts  = line.strip().split("\t")
        left   = int(parts[1])      # absolute genomic position
        right  = int(parts[2]) + 1  # +1 to make half-open interval
        newick = clean_newick(parts[3], sample_names)

        records.append({
            "tree_index": tree_index,
            "left":   float(left),
            "right":  float(right),
            "mid":    float(0.5 * (left + right)),
            "newick": newick,
        })

    return records, seq_start, seq_end


# Read sample names from the sites file header
with open(SITES_FILE) as f:
    sample_names = f.readline().strip().split("\t")[1:]

print(f"Sample names loaded: {len(sample_names)} haplotypes")
print("First few:", sample_names[:6])

# Sanity check — print first cleaned Newick
first_smc = sorted(ARG_SAMPLES_DIR.glob(f"{POPULATION.lower()}.*.smc.gz"))[0]
with gzip.open(first_smc, "rt") as f:
    for line in f:
        if line.startswith("TREE"):
            nhx = line.strip().split("\t")[3]
            print("\nFirst cleaned Newick (first 200 chars):")
            print(clean_newick(nhx, sample_names)[:200])
            break

Sample names loaded: 40 haplotypes
First few: ['NA06984_1', 'NA06984_2', 'NA06985_1', 'NA06985_2', 'NA06986_1', 'NA06986_2']

First cleaned Newick (first 200 chars):
((((NA11832_1:714.120803,NA11830_1:714.120803):973.178029,NA06986_2:1687.298832):30157.072316,((NA10847_2:270.834843,NA07056_1:270.834843):18540.497640,(NA07048_2:11095.665572,((NA11829_1:185.328022,N


In [58]:
# Convert all posterior samples
smc_files = sorted(
    ARG_SAMPLES_DIR.glob(f"{POPULATION.lower()}.*.smc.gz"),
    key=lambda p: int(p.name.split(".")[1])  # sort numerically by iteration
)

print(f"Converting {len(smc_files)} .smc.gz files...\n")

for smc_file in smc_files:
    iteration = smc_file.name.split(".")[1]
    stem = f"{POPULATION.lower()}_sample{iteration}"

    output_tree_path = TREES_DIR / f"{stem}.tree"
    output_csv_path  = CSV_DIR / f"{stem}_breaks.csv"

    # Skip if already converted
    if output_tree_path.exists() and output_csv_path.exists():
        print(f"  [skip] {stem} already converted")
        continue

    records, seq_start, seq_end = smc_to_newick_records(smc_file, sample_names)

    if len(records) == 0:
        print(f"  [warn] No trees found in {smc_file.name}")
        continue

    # Write .tree file (one Newick per line)
    with open(output_tree_path, "w") as f:
        for rec in records:
            f.write(rec["newick"] + "\n")

    # Write _breaks.csv (absolute genomic coordinates)
    with open(output_csv_path, "w", newline="") as cf:
        writer = csv.writer(cf)
        writer.writerow(["file", "tree_index", "left", "right", "mid"])
        for rec in records:
            writer.writerow([stem, rec["tree_index"], rec["left"], rec["right"], rec["mid"]])

    print(
        f"  {stem}.tree  "
        f"({len(records):,} trees | "
        f"[{records[0]['left']:.0f}, {records[-1]['right']:.0f}))"
    )

print("\nAll done.")
print(f"  .tree files → {TREES_DIR}")
print(f"  .csv files  → {CSV_DIR}")

Converting 31 .smc.gz files...

  ceu_sample0.tree  (14,910 trees | [130000000, 140000001))
  ceu_sample10.tree  (13,568 trees | [130000000, 140000001))
  ceu_sample20.tree  (13,292 trees | [130000000, 140000001))
  ceu_sample30.tree  (13,228 trees | [130000000, 140000001))
  ceu_sample40.tree  (12,987 trees | [130000000, 140000001))
  ceu_sample50.tree  (13,051 trees | [130000000, 140000001))
  ceu_sample60.tree  (12,935 trees | [130000000, 140000001))
  ceu_sample70.tree  (13,113 trees | [130000000, 140000001))
  ceu_sample80.tree  (13,056 trees | [130000000, 140000001))
  ceu_sample90.tree  (12,866 trees | [130000000, 140000001))
  ceu_sample100.tree  (13,121 trees | [130000000, 140000001))
  ceu_sample110.tree  (13,106 trees | [130000000, 140000001))
  ceu_sample120.tree  (12,952 trees | [130000000, 140000001))
  ceu_sample130.tree  (12,938 trees | [130000000, 140000001))
  ceu_sample140.tree  (13,079 trees | [130000000, 140000001))
  ceu_sample150.tree  (13,045 trees | [130000000,

## 6. Summary

Quick overview of what was produced.

In [59]:
tree_files = sorted(TREES_DIR.glob("*.tree"))
csv_files  = sorted(CSV_DIR.glob("*_breaks.csv"))

print(f"Region:            chr{CHROM}:{REGION_START:,}–{REGION_END:,}")
print(f"Population:        {POPULATION} ({N_SAMPLES} individuals, {2*N_SAMPLES} haplotypes)")
print(f"MCMC samples:      {len(tree_files)} posterior ARGs")
print()

# Show tree counts per sample
print(f"{'File':<35} {'Trees':>8}")
print("-" * 45)
for tf in tree_files:
    with open(tf) as f:
        n = sum(1 for _ in f)
    print(f"  {tf.name:<33} {n:>8,}")

print()
print("Ready for phylodyn in R:")
print(f"  tree files : {TREES_DIR}/*.tree")
print(f"  coord files: {CSV_DIR}/*_breaks.csv")

Region:            chr2:130,000,000–140,000,000
Population:        CEU (20 individuals, 40 haplotypes)
MCMC samples:      31 posterior ARGs

File                                   Trees
---------------------------------------------
  ceu_sample0.tree                    14,910
  ceu_sample10.tree                   13,568
  ceu_sample100.tree                  13,121
  ceu_sample110.tree                  13,106
  ceu_sample120.tree                  12,952
  ceu_sample130.tree                  12,938
  ceu_sample140.tree                  13,079
  ceu_sample150.tree                  13,045
  ceu_sample160.tree                  12,870
  ceu_sample170.tree                  12,808
  ceu_sample180.tree                  12,779
  ceu_sample190.tree                  12,797
  ceu_sample20.tree                   13,292
  ceu_sample200.tree                  12,899
  ceu_sample210.tree                  12,743
  ceu_sample220.tree                  12,825
  ceu_sample230.tree                  12,912
  c

## Notes on branch lengths and phylodyn

- Branch lengths in the Newick files are in **generations**.
- If phylodyn expects **years**, multiply by a generation time (typically 29 years for humans).
- If phylodyn expects **coalescent units**, divide by 2N (= 20,000 for diploid N=10,000).
- The **burn-in** samples (early MCMC iterations) should be discarded before analysis. With `SAMPLE_STEP=10` and `MCMC_ITERS=200`, a common choice is to discard the first 50–100 iterations (samples 0–50) as burn-in and use samples 100–200 for inference.